# Part 2: Measuring muon selection efficiencies

**Estimated duration: 85 minutes** · CMS 2016 Run G Open Data · ROOT / RDataFrame

In Part 1 you designed selections to identify prompt muons and obtain a clean dimuon sample. A stringent selection removes background, **but also discards signal candidates**. How much of the sample do we keep?

**Learning objectives**

1. Compute **event-level**, conditional and cumulative cut efficiencies.
2. Compute **per-muon** ID and isolation efficiencies within a common kinematic acceptance.
3. Plot efficiencies versus $p_{T}$ and $\eta$ with binomial statistical uncertainties.
4. Discuss selection efficiency versus signal purity and what cannot be inferred from a biased dataset.

**Important:** This exercise uses a preselected 2016 MET sample containing events with muons. Our ratios are *relative selection efficiencies in this sample*, **not absolute CMS reconstruction, identification, or trigger efficiencies**. No simulated truth labels, trigger-unbiased probe sample, or background subtraction are available in this exercise.

## 1. Efficiency: definition and interpretation (10 min)

For a well-defined **denominator** containing $N_{\mathrm{total}}$ candidates, and a subset of $N_{\mathrm{pass}}$ candidates passing a requirement:

$$\varepsilon = \frac{N_{\mathrm{pass}}}{N_{\mathrm{total}}}. $$

For consecutive cuts $A,B,C$, define:

- **Step (conditional) efficiency:** $\varepsilon_{B\mid A}=N(A\cap B)/N(A)$.
- **Cumulative efficiency:** $\varepsilon_{A\cap B\cap C}=N(A\cap B\cap C)/N_{\mathrm{baseline}}$.

The product of successive conditional efficiencies equals the cumulative efficiency. For many trials, an approximate 1-standard-deviation uncertainty is $\sqrt{\varepsilon(1-\varepsilon)/N_{\mathrm{total}}}$. We will instead use ROOT `TEfficiency` with binomial confidence intervals for binned plots.

**Think before computing:** Is $N$ a number of **events** or a number of **muons**? Different choices answer different physics questions.

## 2. Load ROOT and the same dataset as in Part 1 (5 min)

In [ ]:
cd ~/work/CmsOpenData/RDFAnalyzer

In [ ]:
import ROOT as R
from utils import *
import os
os.makedirs("../temp", exist_ok=True)
%jsroot off

R.gROOT.ProcessLine('.L ./_tdrstyle.C')
R.setTDRStyle()
R.gROOT.SetBatch(1)
df = loadRDF("Events", DATA_PATH)

### 2.1 Baseline event count

As a warm-up, use `Count()` to compare the full input to the subset with at least two reconstructed muons. This dataset was already skimmed, so the fraction may be close to 1. It is not an efficiency for detecting all muons produced in pp collisions.

In [ ]:
n_all = df.Count()
df_two = df.Filter("nMuon >= 2")
n_two = df_two.Count()

n_all, n_two = n_all.GetValue(), n_two.GetValue()
print(f"Events in dataset:       {n_all:,}")
print(f"Events with >=2 muons:   {n_two:,}")
print(f"Fraction passing:        {n_two/n_all:.3%}" if n_all else "No events available")

## 3. Event-level efficiency: cumulative cutflow (15 min)

The next code applies successive requirements **to the same muons** and counts events containing **at least two muons** passing all requirements so far. The reference denominator is the number of events with at least two reconstructed muons.

**Default requirements** (change the numbers to test alternatives):

- Kinematics: $p_T>15$ GeV, $|\eta|<2.4$.
- Loose ID, then Medium ID, then Tight ID.
- Tracker relative isolation: $I_{\mathrm{rel}}^{\mathrm{trk}}<0.15$.

A requirement on the muon multiplicity is an *event* selection. Conversely, the Tight ID working point acts on individual muons; requiring at least two passing muons makes it an event selection too. No opposite-charge or mass requirement is applied here.

In [ ]:
# A cumulative mask is built afresh at every stage; all cuts act on the same muon.
steps = [
    ("Kinematics", "(Muon_pt > 15.f) && (abs(Muon_eta) < 2.4f)"),
    ("Loose ID", "Muon_looseId == 1"),
    ("Medium ID", "Muon_mediumId == 1"),
    ("Tight ID", "Muon_tightId == 1"),
    ("Tracker isolation < 0.15", "Muon_tkRelIso < 0.15f"),
]

cutflow = [("At least 2 reconstructed muons", df_two.Count())]
cumulative_masks = []
for label, cut in steps:
    cumulative_masks.append(f"({cut})")
    combined_mask = " && ".join(cumulative_masks)
    cutflow.append((label, df_two.Filter(f"Sum({combined_mask}) >= 2").Count()))

ref = cutflow[0][1].GetValue()
prev = ref
print(f"{'Requirement':31} {'Events':>12} {'Step efficiency':>18} {'Cumulative':>15}")
print("-"*80)
for i, (label, count) in enumerate(cutflow):
    n = count.GetValue()
    eff_step = n/prev if prev else float('nan')
    eff_cum = n/ref if ref else float('nan')
    print(f"{label:31} {n:12,d} {eff_step:18.2%} {eff_cum:15.2%}" if i else f"{label:31} {n:12,d} {'—':>18} {1.:15.2%}")
    prev = n

**Checkpoint A**

1. Which requirement rejects the most events? Which has the smallest *conditional* efficiency?
2. Confirm that the cumulative efficiency after the last cut is the product of all step efficiencies.
3. Repeat the cutflow changing the isolation requirement from 0.15 to **0.10**. What is the effect?
4. Why can two muons individually pass different cuts, yet the event fail the *combined* selection?

**For your report:** Record the complete cutflow table, the chosen isolation threshold, and a short interpretation.

## 4. Per-muon efficiency: ID and isolation (15 min)

Now change the counting unit from **events** to **individual reconstructed muons**. Use the common baseline `Muon_pt > 10 GeV` and `|Muon_eta| < 2.4` for **every** numerator and denominator. This makes the relative ID efficiencies comparable.

For example:

$$\varepsilon_{\mathrm{tight\,ID}} = \frac{N(\mathrm{baseline}\cap\mathrm{tight})}{N(\mathrm{baseline})}, \qquad
\varepsilon_{\mathrm{iso}\mid\mathrm{tight}} = \frac{N(\mathrm{baseline}\cap\mathrm{tight}\cap\mathrm{isolated})}{N(\mathrm{baseline}\cap\mathrm{tight})}. $$

The second quantity is the **isolation efficiency conditional on Tight ID**; do not confuse it with the combined Tight+Iso efficiency relative to the full baseline.

In [ ]:
# Change these definitions to study different working points or isolation thresholds.
muon_definitions = [
    ("Muon_baseMask", "(Muon_pt > 10.f) && (abs(Muon_eta) < 2.4f)"),
    ("Muon_looseMask", "Muon_baseMask && (Muon_looseId == 1)"),
    ("Muon_mediumMask", "Muon_baseMask && (Muon_mediumId == 1)"),
    ("Muon_tightMask", "Muon_baseMask && (Muon_tightId == 1)"),
    ("Muon_tightIsoMask", "Muon_tightMask && (Muon_tkRelIso < 0.15f)"),
]

df_mu = loadRDF("Events", DATA_PATH)
for name, expression in muon_definitions:
    df_mu = df_mu.Define(name, expression)
    df_mu = df_mu.Define(f"n_{name}", f"Sum({name})")

muon_counts = {name: df_mu.Sum(f"n_{name}") for name, _ in muon_definitions}
muon_counts = {name: int(action.GetValue()) for name, action in muon_counts.items()}
n_base = muon_counts["Muon_baseMask"]
print(f"{'Working point':20} {'Muons':>12} {'Baseline efficiency':>22}")
print("-"*59)
for name, n in muon_counts.items():
    print(f"{name:20} {n:12,d} {n/n_base:22.2%}" if n_base else f"{name:20} {n:12,d} {'—':>22}")

n_tight = muon_counts["Muon_tightMask"]
n_tight_iso = muon_counts["Muon_tightIsoMask"]
print(f"\nIsolation efficiency conditional on Tight ID: {n_tight_iso/n_tight:.2%}" if n_tight else "No Tight muons")

**Checkpoint B**

1. Is the Tight ID efficiency larger or smaller than the Loose ID efficiency? Explain.
2. Are the **event** and **muon** efficiency values equal? Why not?
3. Why can `Tight ID + isolation` have a much lower efficiency than Tight ID alone?
4. Is a higher efficiency always better? Think about background rejection and sample purity.

## 5. Binned efficiencies versus $p_T$ and $\eta$ (30 min)

An integrated efficiency hides the dependence on kinematics. A working point may perform differently for low-$p_T$ and forward muons.

**Method**

1. Fill two histograms with exactly the same binning: **all baseline muons** and **baseline muons passing the selection**.
2. Construct `TEfficiency(passed, total)` for each ID working point.
3. Plot efficiency with **68.3% Clopper–Pearson binomial intervals** (not a simple histogram division).

Every passing muon must also be in the denominator. We use **unweighted data counts**, not normalized histograms.

In [ ]:
# Build vector-valued columns; RDataFrame fills one histogram entry per selected muon.
labels = {"base": "Muon_baseMask", "loose": "Muon_looseMask", "medium": "Muon_mediumMask", "tight": "Muon_tightMask", "tight_iso": "Muon_tightIsoMask"}
for label, mask in labels.items():
    df_mu = df_mu.Define(f"eff_{label}_pt", f"Muon_pt[{mask}]")
    df_mu = df_mu.Define(f"eff_{label}_eta", f"Muon_eta[{mask}]")

# Students may change the number of bins, especially if statistical errors are large.
binnings = {"pt": (12, 10., 130.), "eta": (12, -2.4, 2.4)}
xlabels = {"pt": "Muon p_{T} (GeV)", "eta": "Muon #eta"}

hist_actions = {}
for variable, (nbins, xmin, xmax) in binnings.items():
    for label in labels:
        model = (f"h_eff_{label}_{variable}", f";{xlabels[variable]};Muons", nbins, xmin, xmax)
        hist_actions[(variable, label)] = df_mu.Histo1D(model, f"eff_{label}_{variable}")

hist_eff = {key: action.GetValue() for key, action in hist_actions.items()}
print("Filled all numerator and denominator histograms.")

In [ ]:
# The passing distribution is always a subset of the *same* baseline histogram.
colors = {"loose": R.kBlue+1, "medium": R.kGreen+2, "tight": R.kRed+1, "tight_iso": R.kViolet+1}
plot_labels = {"loose": "Loose ID", "medium": "Medium ID", "tight": "Tight ID", "tight_iso": "Tight ID + isolation"}
eff_plots, efficiency_canvases = {}, {}

for variable in binnings:
    R.gROOT.FindObject(f"c_eff_{variable}") and R.gROOT.FindObject(f"c_eff_{variable}").Close()
    canvas = R.TCanvas(f"c_eff_{variable}", "", 850, 620)
    legend = R.TLegend(0.53, 0.17, 0.88, 0.40)
    for i, label in enumerate(colors):
        passed, total = hist_eff[(variable, label)], hist_eff[(variable, "base")]
        if not R.TEfficiency.CheckConsistency(passed, total):
            raise ValueError(f"Incompatible passed/total histograms: {variable}, {label}")
        eff = R.TEfficiency(passed, total)
        eff.SetStatisticOption(R.TEfficiency.kFCP)
        eff.SetConfidenceLevel(0.682689492)
        eff.SetTitle(f";{xlabels[variable]};Selection efficiency")
        eff.SetMarkerStyle(20+i)
        eff.SetMarkerColor(colors[label])
        eff.SetLineColor(colors[label])
        eff.Draw("AP" if i == 0 else "P SAME")
        eff_plots[(variable, label)] = eff  # Keep Python references alive.
        legend.AddEntry(eff, plot_labels[label], "lep")
    canvas.Update()
    eff_plots[(variable, "loose")].GetPaintedGraph().GetYaxis().SetRangeUser(0., 1.08)
    legend.Draw()
    canvas.Draw()
    efficiency_canvases[variable] = (canvas, legend)
    canvas.SaveAs(f"../temp/muon_efficiency_{variable}.png")

### 5.1 Read an efficiency and its asymmetric statistical errors

Use the bin near 35 GeV to see the meaning of the error bars. The *lower* and *upper* uncertainties need not be symmetric, especially near efficiencies of 0 or 1.

In [ ]:
bin_pt = hist_eff[("pt", "base")].GetXaxis().FindBin(35.)
eff_tight_pt = eff_plots[("pt", "tight")]
print(f"pT bin: {hist_eff[('pt', 'base')].GetXaxis().GetBinLowEdge(bin_pt):.1f}–{hist_eff[('pt', 'base')].GetXaxis().GetBinUpEdge(bin_pt):.1f} GeV")
print(f"Total:  {hist_eff[('pt', 'base')].GetBinContent(bin_pt):.0f}")
print(f"Passed: {hist_eff[('pt', 'tight')].GetBinContent(bin_pt):.0f}")
print(f"Tight efficiency = {eff_tight_pt.GetEfficiency(bin_pt):.3f} -{eff_tight_pt.GetEfficiencyErrorLow(bin_pt):.3f} +{eff_tight_pt.GetEfficiencyErrorUp(bin_pt):.3f}")

### 5.2 Isolation efficiency *given* Tight ID

Here the denominator changes: the **Tight ID muons**, not all baseline muons. The numerator is the **Tight + isolated** muons. Compare this curve with `Tight ID + isolation` in the previous plot.

In [ ]:
h_tight = hist_eff[("pt", "tight")]
h_tight_iso = hist_eff[("pt", "tight_iso")]
if not R.TEfficiency.CheckConsistency(h_tight_iso, h_tight):
    raise ValueError("Tight + isolation must be a subset of Tight ID")

R.gROOT.FindObject("c_iso_conditional") and R.gROOT.FindObject("c_iso_conditional").Close()
c_iso = R.TCanvas("c_iso_conditional", "", 850, 620)
eff_iso_given_tight = R.TEfficiency(h_tight_iso, h_tight)
eff_iso_given_tight.SetStatisticOption(R.TEfficiency.kFCP)
eff_iso_given_tight.SetConfidenceLevel(0.682689492)
eff_iso_given_tight.SetTitle(";Muon p_{T} (GeV);Isolation efficiency given Tight ID")
eff_iso_given_tight.SetLineColor(R.kViolet+1)
eff_iso_given_tight.SetMarkerColor(R.kViolet+1)
eff_iso_given_tight.SetMarkerStyle(20)
eff_iso_given_tight.Draw("AP")
c_iso.Update()
eff_iso_given_tight.GetPaintedGraph().GetYaxis().SetRangeUser(0., 1.08)
c_iso.Draw()
c_iso.SaveAs("../temp/muon_isolation_conditional_pt.png")

**Checkpoint C**

1. Describe the shape of $\varepsilon_{\mathrm{tight}}(p_T)$ and $\varepsilon_{\mathrm{tight}}(\eta)$. Is the efficiency constant?
2. Compare the Loose, Medium and Tight curves. Why do they differ?
3. Which phase-space bins have the largest uncertainties? Why?
4. What happens if you double the number of $p_T$ bins (12 $\to$ 24)? Does the underlying selection change?
5. Explain the difference between **Tight+Iso / Baseline** and **Tight+Iso / Tight** using the formulas.

**For your report:** Include at least two figures: efficiency versus $p_T$ and versus $\eta$, with error bars, and a paragraph about the efficiency–purity trade-off.

## 6. Optional extension: how would CMS measure a detector efficiency? (5 min)

What we measured is **not** the probability that a real muon produced in a collision is reconstructed or triggers the detector. The input was already reconstructed, skimmed and recorded using trigger selections.

To measure (for example) an identification efficiency from collision data, CMS often uses **tag-and-probe** with a resonance such as `Z → μμ`:

1. Select a well-identified, triggered **tag** muon.
2. Construct a minimally biased **probe** sample with an independent reference selection.
3. Count probes that pass/fail the identification requirement, usually extracting the signal yield from mass fits in the pass and fail samples.
4. Measure the efficiency in bins of probe kinematics and, for MC comparisons, derive data/MC scale factors.

That would require an appropriate trigger/reference sample and background treatment; the present skim does not supply what is needed for an unbiased measurement. Think about how the next part's Z peak could be used in a future tag-and-probe exercise.

## Before moving to Part 3 (5 min)

You should now be able to answer: **How many muons/events survive each requirement, how does this depend on their kinematics, and why does that matter when measuring a Z resonance?**

Keep your plots and cutflow. You can now use the selected dimuon histogram saved in Part 1 to determine the mass and width of the Z boson.

**Next:** [Part 3 — Measurement of the Z-boson mass and width](./Part3_Z_boson_fit.ipynb).